# ゼロから作るショアのアルゴリズム 第2回: 剰余加算器

[第1回](ja/320_shor_scratch_adder.ipynb)では、通常の量子加算器を作りました。今回はその上に**剰余**加算 $a+b \bmod N$ を実装します — $x^j \bmod N$ を計算する過程で生じるすべての中間値は $[0, N)$ の範囲に収める必要があるため、これがショアのアルゴリズムの中で実際に必要とされる演算です。

参考文献: V. Vedral, A. Barenco, A. Ekert, ["Quantum Networks for Elementary Arithmetic Operations"](https://arxiv.org/abs/quant-ph/9511018) (1995), Figure 3

## 5ステップのアイデア

$(a+b) \bmod N$ は古典的には簡単です: 足し算して、結果が $N$ 以上ならその分だけ $N$ を引きます。しかし量子コンピュータでは、通常のように「確認してから分岐する」ことはできません(それには測定が必要になり、大事にしたい重ね合わせが壊れてしまいます)。代わりに、確認は*量子ビットを使って*行い、条件付きの引き算は*制御回路を使って*行う必要があります。

1. **足し算**: $b \leftarrow a + b$(第1回の加算器を使用)
2. **$N$ を引く**: $b \leftarrow b - N$(加算器を逆向きに実行)
3. **符号を検出**: ステップ2で $b$ が負になった場合、加算器の一番上の余分な量子ビットが $1$ になります(これは借り/オーバーフローのフラグのように働きます) — このビットをフラグ用量子ビット $t$ にコピーします
4. **条件付きで $N$ を足し戻す**: $t=1$ の場合(つまり $a+b < N$ であり、$N$ を引きすぎた場合)、$N$ を足し戻します — ただし重ね合わせの中で $t=1$ になっている枝*だけ*に対して、$t$ を追加の制御量子ビットとして使って行います
5. **後片付け**: $t$ を $\ket 0$ に戻し、符号検出のステップを元に戻します。これにより回路全体が、$b$ 以外には何も影響を残さないクリーンで再利用可能な操作になります

ステップ1〜2は加算器/減算器をそのまま使います。ステップ4には回路全体の*制御版*が必要で、これがここでの新しい面白い部分です。

In [1]:
!pip install git+https://github.com/blueqat/blueqatSDK

/Users/yuichirominato/.pyenv/shims/pip: line 8: /opt/homebrew/opt/pyenv/bin/pyenv: No such file or directory


## 第1回の部品を再利用する

前回と同じ `carry`/`carry_inv`/`sum_gate` に加え、それらを組み立てた加算器全体 `add_circuit`、そしてその逆演算 `sub_circuit` を用意します。`sub_circuit` は、同じゲート列を単に逆順に実行するだけで得られます(ここで使われているゲートはすべて `CX` か `CCX` であり、どちらも自己逆元なので、順序を逆にするだけで十分です — 個々のゲートを反転させる必要はありません)。

In [2]:
from blueqat import Circuit
from blueqat.gate import CXGate, ToffoliGate

def carry(c, a, b, c_next):
    circ = Circuit()
    circ.ccx[a, b, c_next]
    circ.cx[a, b]
    circ.ccx[c, b, c_next]
    return circ

def carry_inv(c, a, b, c_next):
    circ = Circuit()
    circ.ccx[c, b, c_next]
    circ.cx[a, b]
    circ.ccx[a, b, c_next]
    return circ

def sum_gate(c, a, b):
    circ = Circuit()
    circ.cx[a, b]
    circ.cx[c, b]
    return circ

def add_circuit(c, a, b, n):
    """b := a + b, using carry register c. b must have n+1 qubits."""
    circ = Circuit()
    for i in range(n - 1):
        circ += carry(c[i], a[i], b[i], c[i + 1])
    circ += carry(c[n - 1], a[n - 1], b[n - 1], b[n])
    circ.cx[a[n - 1], b[n - 1]]
    circ += sum_gate(c[n - 1], a[n - 1], b[n - 1])
    for i in range(n - 2, -1, -1):
        circ += carry_inv(c[i], a[i], b[i], c[i + 1])
        circ += sum_gate(c[i], a[i], b[i])
    return circ

def sub_circuit(c, a, b, n):
    """b := b - a: the adder's gate list, reversed."""
    add_circ = add_circuit(c, a, b, n)
    reversed_circ = Circuit()
    reversed_circ.ops = list(reversed(add_circ.ops))
    return reversed_circ

## レジスタ構成

剰余加算器全体には $4n+3$ 個の量子ビットが必要です。

- `c[0..n-1]`: 桁上げレジスタ(足し算・引き算のステップで共用)
- `a[0..n-1]`: 1つ目の加数
- `b[0..n]`: 2つ目の加数・結果を保持するレジスタ(第1回と同じ $n+1$ 幅のレジスタで、ステップ3ではその最上位ビットがオーバーフロー/借りのフラグを兼ねる)
- `N[0..n-1]`: 法(定数として読み込む)
- `t`: 「$N$ を足し戻す必要があったか」を表すフラグ用量子ビット1個
- 補助量子ビット1個(以下で3制御ゲートを作るためだけに必要)

## ステップ4の詳細: 非制御の加算器から制御付き加算器を作る

すでに「$b \leftarrow b + N$」を行う `add_circuit(c, N, b, n)` は持っていますが、ステップ4ではこれを*$t=1$のときだけ*適用する必要があります。つまり、この回路に含まれるすべてのゲートを $t$ で制御したバージョンが必要です。`CX` をもう1つの量子ビットで制御すれば `CCX` になり、これは簡単です。しかし `CCX` をもう1つの量子ビットで制御すると**3制御X**になり、このSDKには直接対応するゲートがありません。標準的なテクニックは次の通りです: 補助量子ビットを1つ使い、最初の2つの制御ビットのANDを `CCX` で補助量子ビットに計算し、その補助量子ビットを2つの制御のうちの1つとして通常の `CCX` を適用し、最後に同じ `CCX` をもう一度使って補助量子ビットを $\ket 0$ に戻します。結果として、2制御ゲートと一時的な量子ビット1個だけを使って、もう1つ余分な量子ビットで制御された `CCX` を実現できます。

In [3]:
def build_modulo_adder(input_a, input_b, input_N, n):
    """b := (a + b) mod N. Returns (circuit, b_register_qubit_indices)."""
    num_qubits = 4 * n + 3
    circ = Circuit(num_qubits)

    c = list(range(n))
    a = list(range(n, 2 * n))
    b = list(range(2 * n, 3 * n + 1))
    N = list(range(3 * n + 1, 4 * n + 1))
    t = 4 * n + 1
    ancilla = 4 * n + 2

    # Encode inputs
    for i in range(n):
        if (input_a >> i) & 1:
            circ.x[a[i]]
        if (input_b >> i) & 1:
            circ.x[b[i]]
        if (input_N >> i) & 1:
            circ.x[N[i]]

    # Steps 1-2: b <- a + b, then b <- b - N
    circ += add_circuit(c, a, b, n)
    circ += sub_circuit(c, N, b, n)

    # Step 3: copy the borrow/overflow bit into the flag t
    circ.cx[b[n], t]

    # Step 4: if t == 1, add N back (t-controlled add_circuit, via the ancilla trick above)
    controlled_add = add_circuit(c, N, b, n)
    for g in controlled_add.ops:
        if isinstance(g, CXGate):
            circ.ccx[t, g.targets[0], g.targets[1]]
        elif isinstance(g, ToffoliGate):
            circ.ccx[t, g.targets[0], ancilla]
            circ.ccx[ancilla, g.targets[1], g.targets[2]]
            circ.ccx[t, g.targets[0], ancilla]

    # Step 5: clean up -- undo the borrow/overflow detection and reset t to |0>
    circ += sub_circuit(c, a, b, n)
    circ.x[b[n]]
    circ.cx[b[n], t]
    circ.x[b[n]]
    circ += add_circuit(c, a, b, n)

    return circ, b

## テストする

第1回と同じビット順序の規則を使います: 結果文字列の中で量子ビット `q` は `state[N - 1 - q]` です。

In [4]:
def run_modulo_adder(val_a, val_b, val_N, n):
    circuit, b = build_modulo_adder(val_a, val_b, val_N, n)
    state = circuit.m[:].run(shots=1).most_common(1)[0][0]
    total_qubits = len(state)
    bits = [state[total_qubits - 1 - q] for q in b]  # bits[0] = LSB
    return int("".join(reversed(bits)), 2)

In [5]:
tests = [
    (7, 6, 11, 4),
    (14, 13, 15, 4),
    (3, 5, 9, 4),
    (0, 0, 7, 3),
    (5, 5, 6, 3),
]
for x, y, N, n in tests:
    result = run_modulo_adder(x, y, N, n)
    expected = (x + y) % N
    print(f"({x}+{y}) mod {N} = {result}  (expected {expected}, {'OK' if result == expected else 'MISMATCH'})")

(7+6) mod 11 = 2  (expected 2, OK)


(14+13) mod 15 = 12  (expected 12, OK)


(3+5) mod 9 = 8  (expected 8, OK)


(0+0) mod 7 = 0  (expected 0, OK)


(5+5) mod 6 = 4  (expected 4, OK)


5つのケースすべてが一致しました。このうちいくつかは、両方の分岐を試すように選んでいます: `(7+6) mod 11` は実際に「引いてから足し戻す」補正が必要になる場合です(`7+6=13 \ge 11`)。一方 `(0+0) mod 7` は決して負にならず、ステップ4では何も起こらないはずです — どちらの場合も、同じ回路で正しく動作します。

## 次回予告

足し算と剰余加算ができたので、次のステップは[制御剰余乗算](ja/322_shor_scratch_controlled_multiplier.ipynb)です — ある定数を、量子ビットで制御しながら $N$ を法として掛ける操作を、被乗数をシフトしたコピーを繰り返し剰余加算することで実現します(可逆的に行う筆算式の掛け算です)。これが、最終的に**べき剰余**、そしてアルゴリズム全体を実装するために必要な部品になります。